 # Evaluation Notebook

In [43]:
import json
import pandas as pd
from datetime import datetime
from google import genai

## 1: Setup Gemini Client

In [44]:
client = genai.Client(vertexai=True, project="adsp-s26-reccys", location="us-central1")
MODEL = "gemini-2.5-flash"
print("Gemini client initialized")
print(f"Model: {MODEL}")
print(f"Timestamp: {datetime.now()}")

Gemini client initialized
Model: gemini-2.5-flash
Timestamp: 2026-05-04 23:43:01.051741


## 2: Install Dependencies 

In [45]:
!pip install openpyxl

## 3: Load Dataset

In [46]:
DATASET_PATH = "updated_50_more products.xlsx"

profiles_df = pd.read_excel(DATASET_PATH, sheet_name="User_Profiles", header=3)
products_df = pd.read_excel(DATASET_PATH, sheet_name="Curated_Products", header=3)
gold_bundles_df = pd.read_excel(DATASET_PATH, sheet_name="Gold_Bundles", header=3)
candidates_df = pd.read_excel(DATASET_PATH, sheet_name="Candidate_Labels", header=3)
splits_df = pd.read_excel(DATASET_PATH, sheet_name="Evaluation_Splits", header=3)

print(f"Loaded {len(profiles_df)} user profiles")
print(f"Loaded {len(products_df)} curated products")
print(f"Loaded {len(gold_bundles_df)} gold bundle items")
print(f"Loaded {len(candidates_df)} candidate labels")

print(f"\nProfiles by country:")
print(profiles_df['country'].value_counts().to_string())
print(f"\nProfiles by room type:")
print(profiles_df['room'].value_counts().to_string())
print(f"\nProducts by store:")
print(products_df['store'].value_counts().to_string())

Loaded 50 user profiles
Loaded 225 curated products
Loaded 150 gold bundle items
Loaded 400 candidate labels

Profiles by country:
country
US       30
India    20

Profiles by room type:
room
Living Room    18
Home Office    17
Dining Room    15

Products by store:
store
Amazon              46
IKEA India          32
Pepperfry           30
Urban Ladder        19
West Elm            18
Target              18
Ashley Furniture    18
WoodenStreet        18
Wayfair             16
IKEA US             10


## 4: Build Product Database for Gemini (US + India)

In [47]:
US_STORES = ['Wayfair', 'IKEA US', 'Amazon', 'Ashley Furniture', 'Target', 'West Elm']
INDIA_STORES = ['IKEA India', 'Pepperfry', 'Urban Ladder', 'WoodenStreet']

def build_product_catalog(products_df, style, room_type, country='US'):
    """Filter products by style, room type, and country."""
    if country == 'India':
        filtered = products_df[
            (products_df['style'] == style) &
            (products_df['room_type'] == room_type) &
            (products_df['store'].isin(INDIA_STORES))
        ]
    else:
        filtered = products_df[
            (products_df['style'] == style) &
            (products_df['room_type'] == room_type) &
            (products_df['store'].isin(US_STORES))
        ]

    catalog = []
    for _, product in filtered.iterrows():
        catalog.append({
            "product_id": product['product_id'],
            "name": product['product_name'],
            "category": product['category'],
            "price": float(product['price_usd']),
            "store": product['store'],
            "description": product['description'],
            "material": product.get('material', ''),
            "color": product.get('color', ''),
        })

    return catalog

# Test US
test_us = build_product_catalog(products_df, "Modern Minimalist", "Living Room", "US")
print(f"\nUS catalog (Modern Minimalist, Living Room): {len(test_us)} products")
for p in test_us:
    print(f"  {p['product_id']}: {p['name']} — ${p['price']} ({p['store']})")

# Test India
test_india = build_product_catalog(products_df, "Modern Minimalist", "Living Room", "India")
print(f"\nIndia catalog (Modern Minimalist, Living Room): {len(test_india)} products")
for p in test_india:
    print(f"  {p['product_id']}: {p['name']} — INR {p['price']} ({p['store']})")


US catalog (Modern Minimalist, Living Room): 9 products
  MM-LR-01: Charlie Deep Seater Sofa — $450.0 (Wayfair)
  MM-LR-02: LACK Coffee Table — $49.99 (IKEA US)
  MM-LR-03: Dimmable LED Floor Lamp — $29.99 (Amazon)
  MM-LR-04: Washable Low-Pile Area Rug — $89.99 (Amazon)
  MM-LR-05: KRAGSTA Nesting Tables — $79.99 (IKEA US)
  MM-LR-06: Neutral Throw Pillow Set — $24.99 (Amazon)
  ASH-MM-LR-01: Caladeron Sofa — $599.0 (Ashley Furniture)
  TGT-MM-LR-01: Threshold Flatweave Area Rug 5x7 — $79.99 (Target)
  WE-MM-LR-01: West Elm Parsons Coffee Table — $349.0 (West Elm)

India catalog (Modern Minimalist, Living Room): 4 products
  IKEA-IN-MM-LR-01: KLIPPAN Loveseat — INR 14990.0 (IKEA India)
  PF-MM-LR-01: Mia 2-Seater Sofa — INR 18999.0 (Pepperfry)
  UL-MM-LR-01: Fujiwara Coffee Table — INR 12999.0 (Urban Ladder)
  WS-MM-LR-01: Miso Side Table — INR 4999.0 (WoodenStreet)


## 5: Gemini Recommendation Function

In [48]:
def get_gemini_recommendations(client, profile, product_catalog, model=MODEL):
    """Send user profile and product catalog to Gemini. Gemini picks the best 3."""

    country = profile.get('country', 'US')
    currency = "INR" if country == "India" else "USD"
    currency_symbol = "₹" if country == "India" else "$"

    prompt = f"""You are a home design product recommendation AI called Sage.

USER PROFILE:
- Query: {profile['user_query']}
- Country: {country}
- Room: {profile['room']}
- Style: {profile['style_preferences']}
- Budget: {currency_symbol}{profile['budget_usd']} {currency} (with {profile['budget_flex_pct']*100:.0f}% flexibility)
- Household: {profile['household_context']}
- Priorities: {profile['functional_priorities']}
- Disliked: {profile['disliked_styles_or_items']}
- Must avoid: {profile['negative_constraints']}

AVAILABLE PRODUCTS:
{json.dumps(product_catalog, indent=2)}

TASK:
Pick exactly 3 products from the available products that best match this user's needs.
The total price of all 3 products must stay within the budget ({currency_symbol}{profile['budget_usd']} {currency}).

Respond ONLY with valid JSON in this exact format, no other text:
{{
    "recommended_products": [
        {{
            "product_id": "ID from catalog",
            "product_name": "name",
            "price": price_number,
            "category": "category",
            "store": "store name",
            "relevance_score": score_0_to_100,
            "reason": "why this product matches"
        }}
    ],
    "total_price": total_number,
    "currency": "{currency}",
    "style_match_explanation": "how bundle matches the style",
    "budget_status": "within_budget or over_budget"
}}"""

    try:
        response = client.models.generate_content(
            model=model,
            contents=prompt,
            config={
                "temperature": 0.2,
                "response_mime_type": "application/json",
            }
        )

        result_text = response.text.strip()
        if result_text.startswith("```"):
            result_text = result_text.split("```")[1]
            if result_text.startswith("json"):
                result_text = result_text[4:]
            result_text = result_text.strip()

        return json.loads(result_text)

    except Exception as e:
        print(f"  ERROR for {profile['profile_id']}: {e}")
        return None

## 6: Evaluation Metrics Functions

In [49]:
def evaluate_style_match(recommended_products, profile_style, products_df):
    matches = 0
    total = len(recommended_products)
    for rec in recommended_products:
        pid = rec.get('product_id', '')
        product_row = products_df[products_df['product_id'] == pid]
        if not product_row.empty:
            if product_row.iloc[0]['style'] == profile_style:
                matches += 1
    return matches / total if total > 0 else 0

def evaluate_budget_compliance(recommended_products, budget, flex):
    total = sum(p.get('price', 0) for p in recommended_products)
    max_budget = budget * (1 + flex)
    return {
        "total_price": total,
        "max_budget": max_budget,
        "within_budget": total <= max_budget,
        "budget_usage_pct": (total / budget) * 100
    }

def evaluate_category_coverage(recommended_products, required_categories_str):
    required = [cat.strip().lower() for cat in required_categories_str.split(';')]
    recommended_cats = [p.get('category', '').lower() for p in recommended_products]
    recommended_names = [p.get('product_name', '').lower() for p in recommended_products]
    
    covered = 0
    for req in required:
        req_words = req.split()
        for rec_cat in recommended_cats:
            if any(word in rec_cat for word in req_words) or any(word in req for word in rec_cat.split()):
                covered += 1
                break
        else:
            for name in recommended_names:
                if any(word in name for word in req_words):
                    covered += 1
                    break
    
    return {
        "required": required,
        "recommended": recommended_cats,
        "covered": covered,
        "total_required": len(required),
        "coverage_pct": (covered / len(required)) * 100 if required else 0
    }

def evaluate_constraint_respect(recommended_products, negative_constraints_str, products_df):
    negatives = [n.strip().lower() for n in negative_constraints_str.split(',')]
    violations = []
    for rec in recommended_products:
        name = rec.get('product_name', '').lower()
        for neg in negatives:
            if neg in name:
                violations.append(f"{rec.get('product_name')} violates: {neg}")
    return {
        "violations": violations,
        "violation_count": len(violations),
        "clean": len(violations) == 0
    }

def evaluate_gold_match(recommended_products, gold_bundle, profile_id):
    gold_ids = set(gold_bundle[gold_bundle['profile_id'] == profile_id]['gold_product_id'].tolist())
    rec_ids = set(p.get('product_id', '') for p in recommended_products)
    matches = gold_ids.intersection(rec_ids)
    return {
        "gold_products": list(gold_ids),
        "recommended_products": list(rec_ids),
        "exact_matches": list(matches),
        "match_count": len(matches),
        "match_pct": (len(matches) / len(gold_ids)) * 100 if gold_ids else 0
    }

def calculate_overall_score(style_score, budget_result, coverage_result, constraint_result, gold_result):
    scores = {
        "style_match": style_score * 100 * 0.25,
        "budget_compliance": (100 if budget_result['within_budget'] else 0) * 0.20,
        "category_coverage": coverage_result['coverage_pct'] * 0.20,
        "constraint_respect": (100 if constraint_result['clean'] else 0) * 0.15,
        "gold_match": gold_result['match_pct'] * 0.10,
        "retailer_match": 100 * 0.10,
    }
    total = sum(scores.values())
    return total, scores

## 7: Run Full Evaluation (50 profiles)

In [51]:
print("AgentRec - Full Evaluation")
print(f"Running {len(profiles_df)} profiles (30 US + 20 India)")
print(f"Timestamp: {datetime.now()}")
print("=" * 30)

all_results = []

for idx, profile in profiles_df.iterrows():
    pid = profile['profile_id']
    style = profile['style_preferences']
    room = profile['room']
    country = profile['country']
    currency_symbol = "INR" if country == "India" else "$"

    print(f"Profile {pid}: {profile['profile_name']}")
    print(f"  Country: {country} | Style: {style} | Room: {room} | Budget: {currency_symbol}{profile['budget_usd']}")

    # Step 1: Get product catalog filtered by country
    catalog = build_product_catalog(products_df, style, room, country)

    if not catalog:
        print(f"  WARNING: No products found for {style} + {room} + {country}")
        all_results.append({
            'profile_id': pid, 'profile_name': profile['profile_name'],
            'country': country, 'style': style, 'room': room,
            'budget': profile['budget_usd'], 'status': 'NO_PRODUCTS', 'overall_score': 0
        })
        continue

    print(f"  Products available: {len(catalog)} ({country} stores only)")

    # Step 2: Get Gemini recommendations
    recommendations = get_gemini_recommendations(client, profile, catalog)

    if not recommendations or 'recommended_products' not in recommendations:
        print(f"  FAILED: No recommendations returned")
        all_results.append({
            'profile_id': pid, 'profile_name': profile['profile_name'],
            'country': country, 'style': style, 'room': room,
            'budget': profile['budget_usd'], 'status': 'FAILED', 'overall_score': 0
        })
        continue

    rec_products = recommendations['recommended_products']
    print(f"  Recommendations received: {len(rec_products)} products")

    for rp in rec_products:
        print(f"    -> {rp.get('product_name', 'Unknown')} — {currency_symbol}{rp.get('price', 0)} ({rp.get('store', '')})")

    # Step 3: Evaluate
    style_score = evaluate_style_match(rec_products, style, products_df)
    budget_result = evaluate_budget_compliance(rec_products, profile['budget_usd'], profile['budget_flex_pct'])
    coverage_result = evaluate_category_coverage(rec_products, profile['required_categories'])
    constraint_result = evaluate_constraint_respect(rec_products, profile['negative_constraints'], products_df)
    gold_result = evaluate_gold_match(rec_products, gold_bundles_df, pid)
    overall_score, score_breakdown = calculate_overall_score(
        style_score, budget_result, coverage_result, constraint_result, gold_result
    )

    # Print results
    print(f"\n  Evaluation Results:")
    print(f"    Style Match:        {style_score*100:.0f}%")

    budget_status = "PASS" if budget_result['within_budget'] else "FAIL"
    print(f"    Budget Compliance:  {budget_status} ({currency_symbol}{budget_result['total_price']:.0f} / {currency_symbol}{budget_result['max_budget']:.0f})")

    coverage_covered = coverage_result['covered']
    coverage_total = coverage_result['total_required']
    print(f"    Category Coverage:  {coverage_result['coverage_pct']:.0f}% ({coverage_covered}/{coverage_total})")

    if constraint_result['clean']:
        print(f"    Constraint Respect: CLEAN")
    else:
        violations = constraint_result['violations']
        print(f"    Constraint Respect: VIOLATIONS: {violations}")

    gold_count = gold_result['match_count']
    gold_total = len(gold_result['gold_products'])
    print(f"    Gold Match:         {gold_result['match_pct']:.0f}% ({gold_count}/{gold_total})")
    print(f"    OVERALL SCORE:      {overall_score:.1f}/100")

    # Store results
    all_results.append({
        'profile_id': pid,
        'profile_name': profile['profile_name'],
        'country': country,
        'style': style,
        'room': room,
        'budget': profile['budget_usd'],
        'age_range': profile['age_range'],
        'status': 'SUCCESS',
        'recommended_1': rec_products[0].get('product_name', '') if len(rec_products) > 0 else '',
        'recommended_2': rec_products[1].get('product_name', '') if len(rec_products) > 1 else '',
        'recommended_3': rec_products[2].get('product_name', '') if len(rec_products) > 2 else '',
        'total_price': budget_result['total_price'],
        'budget_max': budget_result['max_budget'],
        'budget_pass': budget_result['within_budget'],
        'style_match_pct': style_score * 100,
        'category_coverage_pct': coverage_result['coverage_pct'],
        'constraint_clean': constraint_result['clean'],
        'gold_match_pct': gold_result['match_pct'],
        'overall_score': overall_score,
    })

AgentRec - Full Evaluation
Running 50 profiles (30 US + 20 India)
Timestamp: 2026-05-04 23:59:57.660037
Profile P001: Modern Minimalist Living Room
  Country: US | Style: Modern Minimalist | Room: Living Room | Budget: $700
  Products available: 9 (US stores only)
  Recommendations received: 3 products
    -> Charlie Deep Seater Sofa — $450.0 (Wayfair)
    -> LACK Coffee Table — $49.99 (IKEA US)
    -> Dimmable LED Floor Lamp — $29.99 (Amazon)

  Evaluation Results:
    Style Match:        100%
    Budget Compliance:  PASS ($530 / $770)
    Category Coverage:  100% (3/3)
    Constraint Respect: CLEAN
    Gold Match:         100% (3/3)
    OVERALL SCORE:      100.0/100
Profile P002: Japandi Zen Living Room
  Country: US | Style: Japandi Zen | Room: Living Room | Budget: $700
  Products available: 9 (US stores only)
  Recommendations received: 3 products
    -> Compact Linen Loveseat — $399.0 (Wayfair)
    -> STOCKHOLM Coffee Table — $199.0 (IKEA US)
    -> Rice Paper Floor Lamp — $29.99

## 8: Summary Report

In [52]:
results_df = pd.DataFrame(all_results)
successful = results_df[results_df['status'] == 'SUCCESS']

print("Evaluation Summary report")
print("=" * 30)

print(f"\nProfiles evaluated: {len(results_df)}")
print(f"Successful: {len(successful)}")
print(f"Failed: {len(results_df) - len(successful)}")

if len(successful) > 0:
    print(f"\n{'─' * 20}")
    print("AGGREGATE METRICS:")
    print(f"  Average Overall Score:     {successful['overall_score'].mean():.1f}/100")
    print(f"  Average Style Match:       {successful['style_match_pct'].mean():.1f}%")
    print(f"  Budget Pass Rate:          {successful['budget_pass'].mean()*100:.1f}%")
    print(f"  Avg Category Coverage:     {successful['category_coverage_pct'].mean():.1f}%")
    print(f"  Constraint Clean Rate:     {successful['constraint_clean'].mean()*100:.1f}%")
    print(f"  Average Gold Match:        {successful['gold_match_pct'].mean():.1f}%")

    print(f"\n{'─' * 20}")
    print("BY COUNTRY:")
    for country in ['US', 'India']:
        country_data = successful[successful['country'] == country]
        if len(country_data) > 0:
            print(f"  {country}: Avg Score {country_data['overall_score'].mean():.1f}/100 ({len(country_data)} profiles)")
            print(f"    Budget Pass: {country_data['budget_pass'].mean()*100:.0f}% | Style Match: {country_data['style_match_pct'].mean():.0f}% | Gold Match: {country_data['gold_match_pct'].mean():.0f}%")

    print(f"\n{'─' * 20}")
    print("BY ROOM TYPE:")
    for room in ['Living Room', 'Dining Room', 'Home Office']:
        room_data = successful[successful['room'] == room]
        if len(room_data) > 0:
            print(f"  {room}: Avg Score {room_data['overall_score'].mean():.1f}/100 ({len(room_data)} profiles)")

    print(f"\n{'─' * 20}")
    print("BY STYLE:")
    for style in successful['style'].unique():
        style_data = successful[successful['style'] == style]
        if len(style_data) > 0:
            print(f"  {style}: Avg Score {style_data['overall_score'].mean():.1f}/100 ({len(style_data)} profiles)")

    print(f"\n{'─' * 20}")
    print("Top 5 Best recommendations:")
    top5 = successful.nlargest(5, 'overall_score')
    for _, row in top5.iterrows():
        print(f"  {row['profile_id']}: {row['profile_name']} ({row['country']}) — Score: {row['overall_score']:.1f}")

    print(f"\nBottom 5 (NEEDS IMPROVEMENT):")
    bottom5 = successful.nsmallest(5, 'overall_score')
    for _, row in bottom5.iterrows():
        print(f"  {row['profile_id']}: {row['profile_name']} ({row['country']}) — Score: {row['overall_score']:.1f}")

Evaluation Summary report

Profiles evaluated: 50
Successful: 50
Failed: 0

────────────────────
AGGREGATE METRICS:
  Average Overall Score:     90.9/100
  Average Style Match:       100.0%
  Budget Pass Rate:          94.0%
  Avg Category Coverage:     85.3%
  Constraint Clean Rate:     100.0%
  Average Gold Match:        50.7%

────────────────────
BY COUNTRY:
  US: Avg Score 94.9/100 (30 profiles)
    Budget Pass: 100% | Style Match: 100% | Gold Match: 67%
  India: Avg Score 85.0/100 (20 profiles)
    Budget Pass: 85% | Style Match: 100% | Gold Match: 27%

────────────────────
BY ROOM TYPE:
  Living Room: Avg Score 91.3/100 (18 profiles)
  Dining Room: Avg Score 91.3/100 (15 profiles)
  Home Office: Avg Score 90.2/100 (17 profiles)

────────────────────
BY STYLE:
  Modern Minimalist: Avg Score 94.2/100 (8 profiles)
  Japandi Zen: Avg Score 92.9/100 (8 profiles)
  Coastal Bright: Avg Score 89.2/100 (8 profiles)
  Mid-Century Warm: Avg Score 91.1/100 (9 profiles)
  Bohemian Eclectic: 

## 9: Save Results

In [53]:
output_path = "evaluation_results_50.xlsx"
results_df.to_excel(output_path, index=False, sheet_name="Evaluation_Results")
print(f"\nResults saved to: {output_path}")
print(f"Evaluation complete at: {datetime.now()}")


Results saved to: evaluation_results_50.xlsx
Evaluation complete at: 2026-05-05 00:22:20.218343


## 10: Visualizations

In [54]:
try:
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    if len(successful) > 0:
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        fig.suptitle('AgentRec — Evaluation Results (50 Profiles)', fontsize=16, fontweight='bold')

        # Chart 1: Overall scores by profile
        ax1 = axes[0, 0]
        colors = ['#0D9488' if s >= 70 else '#EA580C' if s >= 50 else '#DC2626' for s in successful['overall_score']]
        ax1.barh(successful['profile_id'], successful['overall_score'], color=colors)
        ax1.set_xlabel('Overall Score (0-100)')
        ax1.set_title('Overall Score by Profile')
        ax1.axvline(x=70, color='green', linestyle='--', alpha=0.5)
        ax1.tick_params(axis='y', labelsize=6)

        # Chart 2: Scores by country
        ax2 = axes[0, 1]
        country_scores = successful.groupby('country')['overall_score'].mean()
        ax2.bar(country_scores.index, country_scores.values, color=['#2563EB', '#0D9488'])
        ax2.set_ylabel('Average Score')
        ax2.set_title('Average Score by Country')
        ax2.set_ylim(0, 100)

        # Chart 3: Scores by room type
        ax3 = axes[0, 2]
        room_scores = successful.groupby('room')['overall_score'].mean()
        ax3.bar(room_scores.index, room_scores.values, color=['#0D9488', '#2563EB', '#7C3AED'])
        ax3.set_ylabel('Average Score')
        ax3.set_title('Average Score by Room Type')
        ax3.set_ylim(0, 100)

        # Chart 4: Scores by style
        ax4 = axes[1, 0]
        style_scores = successful.groupby('style')['overall_score'].mean()
        ax4.barh(style_scores.index, style_scores.values, color='#0D9488')
        ax4.set_xlabel('Average Score')
        ax4.set_title('Average Score by Style')
        ax4.set_xlim(0, 100)

        # Chart 5: Budget compliance by country
        ax5 = axes[1, 1]
        for country in ['US', 'India']:
            country_data = successful[successful['country'] == country]
            budget_data = [country_data['budget_pass'].sum(), len(country_data) - country_data['budget_pass'].sum()]
            labels = ['Within Budget', 'Over Budget']
            ax5.bar([f'{country}\nWithin', f'{country}\nOver'], budget_data,
                    color=['#0D9488', '#DC2626'])
        ax5.set_title('Budget Compliance by Country')
        ax5.set_ylabel('Number of Profiles')

        # Chart 6: US vs India comparison
        ax6 = axes[1, 2]
        metrics = ['Style Match', 'Budget Pass', 'Coverage', 'Constraints']
        us_data = successful[successful['country'] == 'US']
        india_data = successful[successful['country'] == 'India']
        us_vals = [us_data['style_match_pct'].mean(), us_data['budget_pass'].mean()*100,
                   us_data['category_coverage_pct'].mean(), us_data['constraint_clean'].mean()*100]
        india_vals = [india_data['style_match_pct'].mean(), india_data['budget_pass'].mean()*100,
                      india_data['category_coverage_pct'].mean(), india_data['constraint_clean'].mean()*100]
        x = range(len(metrics))
        ax6.bar([i-0.15 for i in x], us_vals, 0.3, label='US', color='#2563EB')
        ax6.bar([i+0.15 for i in x], india_vals, 0.3, label='India', color='#0D9488')
        ax6.set_xticks(x)
        ax6.set_xticklabels(metrics, fontsize=9)
        ax6.set_ylabel('Percentage')
        ax6.set_title('US vs India Comparison')
        ax6.legend()
        ax6.set_ylim(0, 110)

        plt.tight_layout()
        plt.savefig('evaluation_charts_50.png', dpi=150, bbox_inches='tight')
        plt.show()
        print("Charts saved to: evaluation_charts_50.png")

except ImportError:
    print("matplotlib not available — skipping visualizations")
    print("Install with: pip install matplotlib")

Charts saved to: evaluation_charts_50.png


In [56]:
import time

print("LLM Judge Evaluation")
print(f"Timestamp: {datetime.now()}")
print("=" * 30)

judge_results = []

for result in all_results:
    if result['status'] != 'SUCCESS':
        continue
    
    pid = result['profile_id']
    profile = profiles_df[profiles_df['profile_id'] == pid].iloc[0]
    country = result['country']
    currency = "INR" if country == "India" else "USD"
    currency_symbol = "INR" if country == "India" else "$"
    
    judge_prompt = f"""You are an expert interior design judge. Rate this AI recommendation.

USER PROFILE:
- Query: {profile['user_query']}
- Country: {country}
- Room: {profile['room']}
- Style: {profile['style_preferences']}
- Budget: {currency_symbol}{result['budget']} {currency}
- Household: {profile['household_context']}
- Age: {profile.get('age_range', 'Not specified')}
- Pets: {profile.get('pets', 'None')}
- Accessibility needs: {profile.get('accessibility_needs', 'None')}
- Sustainability: {profile.get('sustainability_preference', 'Not specified')}
- Priorities: {profile['functional_priorities']}
- Disliked: {profile['disliked_styles_or_items']}
- Must avoid: {profile['negative_constraints']}

AI RECOMMENDED:
1. {result['recommended_1']}
2. {result['recommended_2']}
3. {result['recommended_3']}
Total: {result['total_price']}

Rate each criterion 1-5 (1=poor, 5=excellent).
Respond ONLY with valid JSON:
{{
    "style_alignment": score,
    "budget_fit": score,
    "functional_match": score,
    "constraint_respect": score,
    "personalization": score,
    "overall_rating": score,
    "brief_feedback": "one sentence feedback"
}}"""

    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=judge_prompt,
            config={"temperature": 0.2, "response_mime_type": "application/json"}
        )
        
        judge_score = json.loads(response.text.strip())
        judge_score['profile_id'] = pid
        judge_score['profile_name'] = result['profile_name']
        judge_score['country'] = result['country']
        judge_results.append(judge_score)
        
        print(f"{pid}: Style={judge_score['style_alignment']}/5 | Budget={judge_score['budget_fit']}/5 | Function={judge_score['functional_match']}/5 | Constraints={judge_score['constraint_respect']}/5 | Personal={judge_score['personalization']}/5 | Overall={judge_score['overall_rating']}/5")
        print(f"  Feedback: {judge_score['brief_feedback']}")
    
    except Exception as e:
        print(f"{pid}: ERROR — {e}")
    
    time.sleep(3)

if judge_results:
    judge_df = pd.DataFrame(judge_results)
   
    print("LLM Judge Summary")
    print(f"{'=' * 20}")
    print(f"Profiles judged: {len(judge_df)}")
    print(f"Average Style Alignment:   {judge_df['style_alignment'].mean():.1f}/5")
    print(f"Average Budget Fit:        {judge_df['budget_fit'].mean():.1f}/5")
    print(f"Average Functional Match:  {judge_df['functional_match'].mean():.1f}/5")
    print(f"Average Constraint Respect:{judge_df['constraint_respect'].mean():.1f}/5")
    print(f"Average Personalization:   {judge_df['personalization'].mean():.1f}/5")
    print(f"Average Overall Rating:    {judge_df['overall_rating'].mean():.1f}/5")
    
    print(f"\nBy Country:")
    for c in ['US', 'India']:
        c_data = judge_df[judge_df['country'] == c]
        if len(c_data) > 0:
            print(f"  {c}: {c_data['overall_rating'].mean():.1f}/5 ({len(c_data)} profiles)")
    
    judge_df.to_excel("llm_judge_results.xlsx", index=False)

LLM Judge Evaluation
Timestamp: 2026-05-05 00:24:26.810626
P001: Style=4/5 | Budget=5/5 | Function=2/5 | Constraints=5/5 | Personal=3/5 | Overall=3/5
  Feedback: The recommendation respects budget and renter constraints well, but misses the mark on functional storage and the sofa's compactness, while the LACK table might not align with quality preferences.
P002: Style=4/5 | Budget=5/5 | Function=1/5 | Constraints=1/5 | Personal=1/5 | Overall=2/5
  Feedback: The recommendation fails to address critical user needs for scratch-resistant fabrics and hidden storage, despite fitting the budget and general style.
P003: Style=5/5 | Budget=5/5 | Function=5/5 | Constraints=5/5 | Personal=5/5 | Overall=5/5
  Feedback: The AI recommendation perfectly aligns with the user's style, budget, functional needs, and specific preferences for a Coastal Bright living room in a humid climate.
P004: Style=5/5 | Budget=5/5 | Function=5/5 | Constraints=5/5 | Personal=5/5 | Overall=5/5
  Feedback: This AI recomm